# 01 - Descarga de datos meteorológicos de AEMET

Este notebook descarga los datos meteorológicos diarios de las estaciones AEMET situadas dentro de la Demarcación Hidrográfica del Miño-Sil para el periodo 2000-2023.

## Proceso

1. Obtener el listado de todas las estaciones meteorológicas de AEMET.
2. Filtrar las estaciones que caen dentro de la Demarcación Miño-Sil usando el shapefile de la CHMS.
3. Descargar en bucle los datos diarios de cada estación en ventanas de 182 días.
4. Guardar los datos en formato parquet en `data/raw/aemet/`.

## Fuente

API OpenData de AEMET: https://opendata.aemet.es/dist/index.html

## 1. Imports y configuración

In [1]:
import os
import time
from pathlib import Path
from datetime import date, timedelta

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from dotenv import load_dotenv
from tqdm import tqdm

# Cargar API Key desde .env
load_dotenv(Path("..") / ".env")
API_KEY = os.getenv("AEMET_API_KEY")
assert API_KEY, "No se ha encontrado AEMET_API_KEY en .env"

# Rutas
DIR_RAW_AEMET = Path("../data/raw/aemet")
DIR_EXTERNAL = Path("../data/external")
PATH_DEMARCACION = (
    DIR_EXTERNAL
    / "Datos-Cartograficos-Plan-Hidrologico-2022-2027"
    / "Division_Administrativa"
    / "Demarcacion_MiñoSil.shp"
)

DIR_RAW_AEMET.mkdir(parents=True, exist_ok=True)

# Parámetros del periodo de estudio
FECHA_INICIO = date(2000, 1, 1)
FECHA_FIN = date(2023, 12, 31)

# Endpoints AEMET
URL_INVENTARIO_ESTACIONES = "https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/todasestaciones"
URL_DATOS_DIARIOS = "https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini}/fechafin/{fecha_fin}/estacion/{estacion}"

HEADERS = {"api_key": API_KEY, "cache-control": "no-cache"}

print(f"API Key cargada: {API_KEY[:20]}...")
print(f"Periodo de descarga: {FECHA_INICIO} a {FECHA_FIN}")

API Key cargada: eyJhbGciOiJIUzI1NiJ9...
Periodo de descarga: 2000-01-01 a 2023-12-31


## 2. Obtener el inventario de estaciones AEMET

La API de AEMET usa un patrón de dos pasos: la primera llamada devuelve una URL temporal donde están los datos reales.

In [ ]:
def solicitar_datos_aemet(url: str, timeout: int = 30) -> dict | list:
    """Llama a la API de AEMET y descarga el JSON de datos.
    
    La API AEMET devuelve primero una URL temporal en el campo 'datos',
    y hay que hacer una segunda llamada a esa URL para obtener los datos reales.
    """
    resp1 = requests.get(url, headers=HEADERS, timeout=timeout, verify=False)
    resp1.raise_for_status()
    metadata = resp1.json()
    if metadata.get("estado") != 200:
        raise RuntimeError(f"AEMET error: {metadata}")
    url_datos = metadata["datos"]
    resp2 = requests.get(url_datos, timeout=timeout, verify=False)
    resp2.raise_for_status()
    return resp2.json()


# Descargar inventario completo de estaciones
print("Solicitando inventario de estaciones AEMET...")
inventario = solicitar_datos_aemet(URL_INVENTARIO_ESTACIONES)
df_estaciones = pd.DataFrame(inventario)
print(f"Estaciones totales AEMET: {len(df_estaciones)}")
df_estaciones.head()

## 3. Filtrar estaciones dentro de la Demarcación Miño-Sil

Las coordenadas de AEMET vienen en formato sexagesimal con orientación (por ejemplo `430649N`, `080244W`). Hay que convertirlas a decimal antes de hacer el filtro espacial.

In [3]:
def coord_sexagesimal_a_decimal(coord: str) -> float:
    """Convierte una coordenada AEMET del tipo '430649N' a decimal.
    
    Los dos últimos dígitos antes de la letra son segundos, los dos anteriores
    minutos y el resto grados. La letra indica orientación (N/S para latitud,
    E/W para longitud).
    """
    if pd.isna(coord) or not isinstance(coord, str):
        return None
    orientacion = coord[-1]
    numero = coord[:-1]
    segundos = int(numero[-2:])
    minutos = int(numero[-4:-2])
    grados = int(numero[:-4])
    decimal = grados + minutos / 60 + segundos / 3600
    if orientacion in ("S", "W"):
        decimal = -decimal
    return decimal


# Convertir coordenadas a decimal
df_estaciones["lat_dec"] = df_estaciones["latitud"].apply(coord_sexagesimal_a_decimal)
df_estaciones["lon_dec"] = df_estaciones["longitud"].apply(coord_sexagesimal_a_decimal)

# Cargar shapefile de la Demarcación Miño-Sil
demarcacion = gpd.read_file(PATH_DEMARCACION)
print(f"CRS demarcación: {demarcacion.crs}")

# Convertir estaciones a GeoDataFrame en WGS84 y reproyectar al CRS de la demarcación
gdf_estaciones = gpd.GeoDataFrame(
    df_estaciones,
    geometry=gpd.points_from_xy(df_estaciones["lon_dec"], df_estaciones["lat_dec"]),
    crs="EPSG:4326",
).to_crs(demarcacion.crs)

# Filtrado espacial: quedarnos solo con las estaciones dentro de la demarcación
estaciones_minosil = gpd.sjoin(gdf_estaciones, demarcacion, predicate="within").drop(columns="index_right")

print(f"Estaciones AEMET dentro de la Demarcación Miño-Sil: {len(estaciones_minosil)}")
estaciones_minosil[["indicativo", "nombre", "provincia", "altitud", "lat_dec", "lon_dec"]].head(10)

CRS demarcación: EPSG:25829
Estaciones AEMET dentro de la Demarcación Miño-Sil: 24


,indicativo,nombre,provincia,altitud,lat_dec,lon_dec
274,1446X,MONTERROSO,LUGO,680,42.809722,-7.768333
290,1505,LUGO AEROPUERTO,LUGO,442,43.111111,-7.457500
291,1518A,LUGO,LUGO,442,42.998333,-7.552500
292,1521I,O PÁRAMO,LUGO,403,42.845278,-7.499444
294,1541B,VILLABLINO,LEON,958,42.928889,-6.333889
295,1542,PUERTO DE LEITARIEGOS,ASTURIAS,1530,42.994167,-6.414167
296,1549,PONFERRADA,LEON,532,42.563889,-6.600000
297,1561I,VEGA DE ESPINAREDA,LEON,770,42.766111,-6.673611
298,1583X,O BARCO DE VALDEORRAS,OURENSE,315,42.415556,-6.993056
299,1631E,A POBRA DE TRIVES,OURENSE,840,42.339444,-7.282500


## 4. Guardar el inventario filtrado

Antes de descargar datos, guardamos el listado de estaciones a procesar. Así, si la descarga se interrumpe, sabemos qué estaba pendiente.

In [4]:
# Guardar inventario de estaciones filtradas
inventario_path = DIR_RAW_AEMET / "inventario_estaciones_minosil.parquet"
estaciones_minosil.drop(columns="geometry").to_parquet(inventario_path, index=False)
print(f"Inventario guardado en {inventario_path}")
print(f"Total estaciones a procesar: {len(estaciones_minosil)}")

Inventario guardado en ..\data\raw\aemet\inventario_estaciones_minosil.parquet
Total estaciones a procesar: 24


## 5. Descarga de datos diarios en bucle

Para cada estación se hacen solicitudes en ventanas de 182 días (límite de AEMET). Cada estación producirá aproximadamente 48 solicitudes para cubrir el periodo 2000-2023.

**Precauciones:**

- Se guarda cada estación en un fichero independiente para poder reanudar tras cortes.
- Si la estación ya está descargada, se salta.
- Se respetan pausas entre llamadas para no saturar la API.

In [6]:
import warnings
import urllib3

# Silenciar warnings de HTTPS
warnings.filterwarnings("ignore", category=urllib3.exceptions.InsecureRequestWarning)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


def descargar_estacion(indicativo: str, fecha_ini: date, fecha_fin: date) -> pd.DataFrame:
    """Descarga los datos diarios de una estación AEMET.
    
    Maneja:
    - 404: la estación no tiene datos en ese periodo (situación normal, no error).
    - 429: rate limit, espera 60 segundos y reintenta.
    - Otros errores: los reporta y continúa.
    """
    dfs = []
    ventanas_sin_datos = 0
    ventanas_con_datos = 0
    
    for f_ini, f_fin in rango_ventanas(fecha_ini, fecha_fin):
        url = URL_DATOS_DIARIOS.format(
            fecha_ini=f"{f_ini.isoformat()}T00:00:00UTC",
            fecha_fin=f"{f_fin.isoformat()}T23:59:59UTC",
            estacion=indicativo,
        )
        for intento in range(3):  # hasta 3 intentos por ventana
            try:
                datos = solicitar_datos_aemet(url)
                if datos:
                    dfs.append(pd.DataFrame(datos))
                    ventanas_con_datos += 1
                break  # éxito, siguiente ventana
            except RuntimeError as e:
                mensaje = str(e)
                if "404" in mensaje or "No hay datos" in mensaje:
                    ventanas_sin_datos += 1
                    break  # sin datos, no reintentar
                else:
                    print(f"    Error inesperado en {indicativo} {f_ini}: {e}")
                    break
            except requests.HTTPError as e:
                if e.response.status_code == 429:
                    print(f"    Rate limit alcanzado. Esperando 60 segundos...")
                    time.sleep(60)
                    continue  # reintentar
                else:
                    print(f"    HTTP error en {indicativo} {f_ini}: {e}")
                    break
            except Exception as e:
                print(f"    Error en {indicativo} {f_ini}: {e}")
                break
        
        time.sleep(2)  # pausa más conservadora entre ventanas
    
    resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    return resultado, ventanas_con_datos, ventanas_sin_datos


# Bucle principal
indicativos = estaciones_minosil["indicativo"].tolist()
print(f"Descargando {len(indicativos)} estaciones...\n")

for indicativo in tqdm(indicativos, desc="Estaciones"):
    salida = DIR_RAW_AEMET / f"estacion_{indicativo}.parquet"
    if salida.exists():
        continue
    
    df, con_datos, sin_datos = descargar_estacion(indicativo, FECHA_INICIO, FECHA_FIN)
    
    if not df.empty:
        df.to_parquet(salida, index=False)
        tqdm.write(f"  {indicativo}: {len(df)} registros ({con_datos} ventanas con datos)")
    else:
        tqdm.write(f"  {indicativo}: SIN DATOS ({sin_datos} ventanas vacías)")

print("\nDescarga completada.")

Descargando 24 estaciones...



Estaciones:   0%|          | 0/24 [00:00<?, ?it/s]

    Rate limit alcanzado. Esperando 60 segundos...
    HTTP error en 1446X 2011-12-17: 500 Server Error: Internal Server Error for url: https://opendata.aemet.es/opendata/sh/caacd323


Estaciones:   4%|▍         | 1/24 [03:38<1:23:37, 218.17s/it]

  1446X: 5739 registros (36 ventanas con datos)
    Rate limit alcanzado. Esperando 60 segundos...
    Rate limit alcanzado. Esperando 60 segundos...
    Rate limit alcanzado. Esperando 60 segundos...
    Rate limit alcanzado. Esperando 60 segundos...
    Rate limit alcanzado. Esperando 60 segundos...
    Rate limit alcanzado. Esperando 60 segundos...


...

  1730E: 8744 registros (49 ventanas con datos)
    Rate limit alcanzado. Esperando 60 segundos...


Estaciones:  96%|█████████▌| 23/24 [2:46:12<07:11, 431.35s/it]

  1735X: 7422 registros (47 ventanas con datos)
    HTTP error en 1738U 2015-12-12: 500 Server Error: Internal Server Error for url: https://opendata.aemet.es/opendata/sh/67084f30
    Rate limit alcanzado. Esperando 60 segundos...


Estaciones: 100%|██████████| 24/24 [2:50:02<00:00, 425.12s/it]

  1738U: 8287 registros (48 ventanas con datos)

Descarga completada.


## 6. Consolidación de resultados

Verificar cuántas estaciones se han descargado con éxito y unificar los datos en un solo fichero.

In [7]:
# Contar estaciones descargadas
ficheros_estacion = list(DIR_RAW_AEMET.glob("estacion_*.parquet"))
print(f"Estaciones descargadas: {len(ficheros_estacion)} / {len(indicativos)}")

# Consolidar en un único parquet
dfs_todos = []
for f in ficheros_estacion:
    df = pd.read_parquet(f)
    dfs_todos.append(df)

if dfs_todos:
    aemet_completo = pd.concat(dfs_todos, ignore_index=True)
    salida_final = Path("../data/interim/aemet_minosil_2000_2023.parquet")
    salida_final.parent.mkdir(parents=True, exist_ok=True)
    aemet_completo.to_parquet(salida_final, index=False)
    print(f"Consolidado guardado en {salida_final}")
    print(f"Registros totales: {len(aemet_completo)}")
    print(f"Columnas: {list(aemet_completo.columns)}")

Estaciones descargadas: 23 / 24
Consolidado guardado en ..\data\interim\aemet_minosil_2000_2023.parquet
Registros totales: 165676
Columnas: ['fecha', 'indicativo', 'nombre', 'provincia', 'altitud', 'tmed', 'prec', 'tmin', 'horatmin', 'tmax', 'horatmax', 'pintMax', 'horaPIntMax', 'hrMedia', 'hrMax', 'horaHrMax', 'hrMin', 'horaHrMin', 'dir', 'velmedia', 'racha', 'horaracha', 'sol', 'presMax', 'horaPresMax', 'presMin', 'horaPresMin']
